In [ ]:
# GPT-2 FineWeb pretraining using the reviewed repository implementation
!nvidia-smi
import torch

num_gpus = max(1, torch.cuda.device_count())
print(f"GPU count: {num_gpus}")
for index in range(num_gpus):
    print(torch.cuda.get_device_name(index))
if num_gpus != 2:
    raise RuntimeError(f"This notebook requires exactly two T4 GPUs, found {num_gpus}.")


In [ ]:
# Install exactly the branch that contains this recipe.
!pip install -q uv
!git clone --depth 1 --branch spirlness/feat/gpt2-fineweb-training https://github.com/spirlness/Automodel.git Automodel
%cd Automodel
!uv sync --locked --group dev --extra fa --inexact


In [ ]:
# Produce the binary dataset with the repository-owned preprocessor.
!uv run python projects/gpt2_fineweb_500m/tools/nanogpt_data_processor.py \
  --dataset HuggingFaceFW/fineweb \
  --set-name sample-10BT \
  --output-dir /kaggle/working/fineweb_1B \
  --max-tokens 1B


In [ ]:
# Train 1B tokens on two T4 GPUs. Use a 16-sample micro-batch on each
# GPU and a global batch of 32, so no gradient accumulation is required.
# Checkpoint retention is enforced by the project-owned checkpoint lifecycle.
!uv run automodel projects/gpt2_fineweb_500m/config/gpt2_fineweb_500m.yaml \
  --nproc-per-node 2 \
  --dataset.file_pattern=/kaggle/working/fineweb_1B_max_tokens_1B/dataset.bin \
  --step_scheduler.global_batch_size=32 \
  --step_scheduler.local_batch_size=16 \
  --step_scheduler.max_steps=30517 \
  --step_scheduler.ckpt_every_steps=10000 \
  --step_scheduler.save_checkpoint_every_epoch=false \
  --checkpoint.checkpoint_dir=/kaggle/working/checkpoints \
  --checkpoint.max_recent_checkpoints=3 \
  --model.torch_dtype=float16 \
  --distributed.mp_policy.param_dtype=torch.float16 \
  --distributed.mp_policy.output_dtype=torch.float16
